In [7]:
from pathlib import Path
import numpy as np
import rasterio

_ROOT = Path("/Users/farhanhassan/makeathon-challenge-2026")
FEAT_DIR = _ROOT / "data" / "cleaned" / "features"
OUT_DIR  = _ROOT / "data" / "cleaned" / "features_slim"

KEEP_BANDS = [
    # AEF summary features (10)
    "aef_cosine_dist_2021", "aef_cosine_dist_2022", "aef_cosine_dist_2023",
    "aef_cosine_dist_2024", "aef_cosine_dist_2025",
    "aef_delta_norm_2021", "aef_delta_norm_2022", "aef_delta_norm_2023",
    "aef_delta_norm_2024", "aef_delta_norm_2025",
    # S1 useful features (3)
    "s1_change_mean", "s1_change_ratio", "s1_after_std",
]

print(f"Keeping {len(KEEP_BANDS)} bands")
print(KEEP_BANDS)

Keeping 13 bands
['aef_cosine_dist_2021', 'aef_cosine_dist_2022', 'aef_cosine_dist_2023', 'aef_cosine_dist_2024', 'aef_cosine_dist_2025', 'aef_delta_norm_2021', 'aef_delta_norm_2022', 'aef_delta_norm_2023', 'aef_delta_norm_2024', 'aef_delta_norm_2025', 's1_change_mean', 's1_change_ratio', 's1_after_std']


In [8]:
import geopandas as gpd

meta_dir = _ROOT / "data" / "concept_data" / "metadata"
train_tiles = gpd.read_file(meta_dir / "train_tiles.geojson")["name"].tolist()
test_tiles  = gpd.read_file(meta_dir / "test_tiles.geojson")["name"].tolist()

print(f"Train tiles: {len(train_tiles)} — {train_tiles}")
print(f"Test tiles:  {len(test_tiles)} — {test_tiles}")

Train tiles: 16 — ['47QMB_0_8', '48PWV_7_8', '48PUT_0_8', '48QWD_2_2', '48PXC_7_7', '48PYB_3_6', '47QQV_2_4', '48QVE_3_0', '18NXH_6_8', '18NWG_6_6', '18NWJ_8_9', '18NXJ_7_6', '18NWH_1_4', '18NYH_9_9', '19NBD_4_4', '18NWM_9_4']
Test tiles:  5 — ['18NVJ_1_6', '18NYH_2_1', '33NTE_5_1', '47QMA_6_2', '48PWA_0_6']


In [9]:
def create_slim(tile_id: str, split: str):
    """Extract KEEP_BANDS from full feature file, save slim version."""
    src_path = FEAT_DIR / split / f"{tile_id}_features.tif"
    dst_dir  = OUT_DIR / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_path = dst_dir / f"{tile_id}_slim.tif"

    with rasterio.open(src_path) as src:
        descs = list(src.descriptions)
        # Map band name -> 1-based index
        name_to_idx = {d: i + 1 for i, d in enumerate(descs) if d}

        # Resolve indices for the bands we want
        indices = []
        found = []
        for b in KEEP_BANDS:
            if b in name_to_idx:
                indices.append(name_to_idx[b])
                found.append(b)
            else:
                print(f"  ⚠️  {b} not found in {tile_id}")

        data = src.read(indices)  # shape (n_bands, H, W)

        profile = src.profile.copy()
        profile.update(count=len(indices), compress="deflate", predictor=2)

        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(data)
            dst.descriptions = tuple(found)

    mb = dst_path.stat().st_size / 1e6
    print(f"  ✅ {tile_id} ({split}): {len(found)} bands, {mb:.1f} MB")
    return dst_path

In [10]:
# Only process tiles that have feature files
avail_train = [t for t in train_tiles if (FEAT_DIR / "train" / f"{t}_features.tif").exists()]
avail_test  = [t for t in test_tiles  if (FEAT_DIR / "test"  / f"{t}_features.tif").exists()]
print(f"Available: {len(avail_train)} train, {len(avail_test)} test")

print("\n=== Processing train tiles ===")
for t in avail_train:
    create_slim(t, "train")

print("\n=== Processing test tiles ===")
for t in avail_test:
    create_slim(t, "test")

print("\nDone!")

Available: 5 train, 5 test

=== Processing train tiles ===
  ✅ 18NXH_6_8 (train): 13 bands, 11.6 MB
  ✅ 18NWG_6_6 (train): 13 bands, 43.3 MB
  ✅ 18NWJ_8_9 (train): 13 bands, 43.5 MB
  ✅ 18NWH_1_4 (train): 13 bands, 43.7 MB
  ✅ 18NWM_9_4 (train): 13 bands, 44.1 MB

=== Processing test tiles ===
  ✅ 18NVJ_1_6 (test): 13 bands, 42.7 MB
  ✅ 18NYH_2_1 (test): 13 bands, 43.8 MB
  ✅ 33NTE_5_1 (test): 13 bands, 43.6 MB
  ✅ 47QMA_6_2 (test): 13 bands, 38.9 MB
  ✅ 48PWA_0_6 (test): 13 bands, 41.6 MB

Done!


In [11]:
# Quick sanity check on one file
sample = list((OUT_DIR / "train").glob("*_slim.tif"))[0]
with rasterio.open(sample) as src:
    print(f"File: {sample.name}")
    print(f"Shape: {src.count} bands × {src.height} × {src.width}")
    print(f"CRS: {src.crs}")
    print(f"Bands: {list(src.descriptions)}")
    data = src.read()
    print(f"dtype: {data.dtype}, range: [{np.nanmin(data):.4f}, {np.nanmax(data):.4f}]")
    print(f"NaN pixels: {np.isnan(data).any(axis=0).sum()} / {src.height * src.width}")
    print(f"File size: {sample.stat().st_size / 1e6:.1f} MB")

# Total disk usage
total = sum(f.stat().st_size for f in OUT_DIR.rglob("*_slim.tif"))
print(f"\nTotal disk: {total / 1e6:.0f} MB")

File: 18NWG_6_6_slim.tif
Shape: 13 bands × 1004 × 998
CRS: EPSG:4326
Bands: ['aef_cosine_dist_2021', 'aef_cosine_dist_2022', 'aef_cosine_dist_2023', 'aef_cosine_dist_2024', 'aef_cosine_dist_2025', 'aef_delta_norm_2021', 'aef_delta_norm_2022', 'aef_delta_norm_2023', 'aef_delta_norm_2024', 'aef_delta_norm_2025', 's1_change_mean', 's1_change_ratio', 's1_after_std']
dtype: float32, range: [-9.7839, 7.0489]
NaN pixels: 2001 / 1001992
File size: 43.3 MB

Total disk: 397 MB


## Validation Tests
Load slim features + RADD labels, check per-band stats, cross-correlation, and class separability.

In [12]:
from rasterio.warp import reproject, Resampling
from scipy.stats import mannwhitneyu

LABEL_DIR = _ROOT / "data" / "concept_data" / "labels" / "train"

# Collect all pixels across train tiles
all_feats = {b: [] for b in KEEP_BANDS}
all_labels = []

for tile in avail_train:
    with rasterio.open(OUT_DIR / "train" / f"{tile}_slim.tif") as src:
        descs = list(src.descriptions)
        H, W = src.height, src.width
        ft, fc = src.transform, src.crs
        for i, d in enumerate(descs):
            all_feats[d].append(src.read(i + 1).ravel())

    # Reproject RADD labels
    radd = np.zeros((H, W), dtype=np.int32)
    with rasterio.open(LABEL_DIR / "radd" / f"radd_{tile}_labels.tif") as lsrc:
        reproject(lsrc.read(1).astype(np.int32), radd,
                  src_transform=lsrc.transform, src_crs=lsrc.crs,
                  dst_transform=ft, dst_crs=fc, resampling=Resampling.nearest)
    all_labels.append((radd > 0).astype(np.int32).ravel())

labels = np.concatenate(all_labels)
feats = {k: np.concatenate(v) for k, v in all_feats.items()}

# Filter valid
valid = np.ones(len(labels), dtype=bool)
for v in feats.values():
    valid &= np.isfinite(v)
labels = labels[valid]
feats = {k: v[valid] for k, v in feats.items()}

print(f"Pixels: {len(labels):,}  |  Deforested: {(labels==1).sum():,} ({100*(labels==1).mean():.1f}%)  |  Stable: {(labels==0).sum():,}")

Pixels: 3,997,169  |  Deforested: 494,423 (12.4%)  |  Stable: 3,502,746


In [13]:
# --- Test 1: Per-band AUC (Mann-Whitney) ---
rng = np.random.default_rng(42)
n = min(10_000, (labels==1).sum(), (labels==0).sum())
d_idx = rng.choice(np.where(labels==1)[0], n, replace=False)
s_idx = rng.choice(np.where(labels==0)[0], n, replace=False)

print(f"{'Band':<30} {'Defor μ':>9} {'Stable μ':>9} {'Δ':>8} {'AUC':>6}")
print("-" * 65)
aucs = {}
for name in KEEP_BANDS:
    d, s = feats[name][d_idx], feats[name][s_idx]
    u, _ = mannwhitneyu(d, s, alternative='two-sided')
    auc = max(u / (n * n), 1 - u / (n * n))
    aucs[name] = auc
    print(f"{name:<30} {d.mean():>9.4f} {s.mean():>9.4f} {d.mean()-s.mean():>8.4f} {auc:>6.3f}")

Band                             Defor μ  Stable μ        Δ    AUC
-----------------------------------------------------------------
aef_cosine_dist_2021              0.1599    0.0456   0.1143  0.675
aef_cosine_dist_2022              0.3670    0.0888   0.2782  0.820
aef_cosine_dist_2023              0.3698    0.0757   0.2941  0.843
aef_cosine_dist_2024              0.4231    0.0784   0.3447  0.922
aef_cosine_dist_2025              0.4556    0.0704   0.3852  0.941
aef_delta_norm_2021               0.4801    0.2863   0.1938  0.675
aef_delta_norm_2022               0.7788    0.3972   0.3815  0.820
aef_delta_norm_2023               0.7807    0.3611   0.4196  0.843
aef_delta_norm_2024               0.8643    0.3665   0.4977  0.922
aef_delta_norm_2025               0.8993    0.3314   0.5679  0.941
s1_change_mean                   -1.0184   -0.1919  -0.8264  0.753
s1_change_ratio                   1.1747    1.0390   0.1357  0.748
s1_after_std                      1.7152    1.5729   0.1423  0.

In [14]:
# --- Test 2: Cross-correlation matrix (redundancy check) ---
import pandas as pd

feat_df = pd.DataFrame({k: feats[k][rng.choice(len(labels), 50_000, replace=False)] for k in KEEP_BANDS})
corr = feat_df.corr()

# Show only pairs with |r| > 0.8 (potential redundancy)
print("=== High correlations (|r| > 0.8) — potential redundancy ===")
found_high = False
for i, a in enumerate(KEEP_BANDS):
    for b in KEEP_BANDS[i+1:]:
        r = corr.loc[a, b]
        if abs(r) > 0.8:
            print(f"  {a} ↔ {b}: r = {r:.3f}")
            found_high = True
if not found_high:
    print("  None — all bands are non-redundant ✅")

print("\n=== Full correlation matrix ===")
# Short names for display
short = [b.replace("aef_cosine_dist_", "cos").replace("aef_delta_norm_", "norm").replace("s1_", "") for b in KEEP_BANDS]
corr_display = corr.copy()
corr_display.index = short
corr_display.columns = short
print(corr_display.round(2).to_string())

=== High correlations (|r| > 0.8) — potential redundancy ===
  None — all bands are non-redundant ✅

=== Full correlation matrix ===
              cos2021  cos2022  cos2023  cos2024  cos2025  norm2021  norm2022  norm2023  norm2024  norm2025  change_mean  change_ratio  after_std
cos2021          1.00    -0.01     0.00     0.00    -0.01      0.00      0.00      0.00      0.00     -0.01        -0.00         -0.00      -0.00
cos2022         -0.01     1.00    -0.00    -0.00     0.00      0.00      0.00      0.00      0.01     -0.01         0.01         -0.01       0.00
cos2023          0.00    -0.00     1.00     0.00    -0.00      0.00      0.00     -0.00     -0.00     -0.00        -0.00          0.00      -0.01
cos2024          0.00    -0.00     0.00     1.00     0.00      0.01      0.00     -0.01      0.00     -0.00        -0.00         -0.00       0.00
cos2025         -0.01     0.00    -0.00     0.00     1.00      0.00      0.01      0.01     -0.01     -0.00        -0.00         -0.00   

In [15]:
# --- Test 3: Per-tile consistency (does signal hold across all tiles?) ---
print(f"{'Tile':<15} {'cos25 AUC':>10} {'norm25 AUC':>11} {'s1_chg AUC':>11} {'Defor %':>8}")
print("-" * 58)

start = 0
for tile in avail_train:
    with rasterio.open(OUT_DIR / "train" / f"{tile}_slim.tif") as src:
        n_pix = src.height * src.width

    tile_lab = labels[start:start+n_pix] if start+n_pix <= len(labels) else None
    if tile_lab is None or (tile_lab==1).sum() < 100:
        start += n_pix
        continue

    d_i = np.where(tile_lab == 1)[0] + start
    s_i = np.where(tile_lab == 0)[0] + start
    k = min(5000, len(d_i), len(s_i))
    d_sub = rng.choice(d_i, k, replace=False)
    s_sub = rng.choice(s_i, k, replace=False)

    tile_aucs = {}
    for band in ["aef_cosine_dist_2025", "aef_delta_norm_2025", "s1_change_mean"]:
        u, _ = mannwhitneyu(feats[band][d_sub], feats[band][s_sub], alternative='two-sided')
        tile_aucs[band] = max(u/(k*k), 1-u/(k*k))

    pct = 100 * (tile_lab==1).mean()
    print(f"{tile:<15} {tile_aucs['aef_cosine_dist_2025']:>10.3f} {tile_aucs['aef_delta_norm_2025']:>11.3f} {tile_aucs['s1_change_mean']:>11.3f} {pct:>7.1f}%")
    start += n_pix

Tile             cos25 AUC  norm25 AUC  s1_chg AUC  Defor %
----------------------------------------------------------
18NXH_6_8            0.916       0.916       0.686    34.5%
18NWG_6_6            0.935       0.935       0.703     6.5%
18NWJ_8_9            0.922       0.922       0.575     3.3%


In [16]:
# --- Test 4: Quick logistic regression baseline (is the combo learnable?) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import cross_val_predict

# Subsample for speed
n_sub = min(100_000, len(labels))
idx = rng.choice(len(labels), n_sub, replace=False)
X = np.column_stack([feats[b][idx] for b in KEEP_BANDS])
y = labels[idx]

print(f"Logistic regression on {n_sub:,} pixels, {X.shape[1]} features")
lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
y_pred = cross_val_predict(lr, X, y, cv=5, method='predict_proba')[:, 1]
y_cls = (y_pred > 0.5).astype(int)

auc = roc_auc_score(y, y_pred)
f1 = f1_score(y, y_cls)
print(f"  5-fold CV  AUC: {auc:.3f}")
print(f"  5-fold CV  F1:  {f1:.3f}")

# Feature-ablation: AEF-only vs S1-only vs combined
aef_cols = [i for i, b in enumerate(KEEP_BANDS) if b.startswith("aef_")]
s1_cols  = [i for i, b in enumerate(KEEP_BANDS) if b.startswith("s1_")]

for label_name, cols in [("AEF only", aef_cols), ("S1 only", s1_cols), ("Combined", list(range(len(KEEP_BANDS))))]:
    Xsub = X[:, cols]
    lr2 = LogisticRegression(max_iter=500, class_weight='balanced', random_state=42)
    yp = cross_val_predict(lr2, Xsub, y, cv=5, method='predict_proba')[:, 1]
    print(f"  {label_name:>10}: AUC = {roc_auc_score(y, yp):.3f}  F1 = {f1_score(y, (yp>0.5).astype(int)):.3f}")

Logistic regression on 100,000 pixels, 13 features
  5-fold CV  AUC: 0.948
  5-fold CV  F1:  0.677
    AEF only: AUC = 0.947  F1 = 0.675
     S1 only: AUC = 0.763  F1 = 0.411
    Combined: AUC = 0.948  F1 = 0.677
